# Field Scan : DeepForest entraîné sur Count banana plants + ReforesTree, testé sur Manuel (Kaggle)

**Question** : en mélangeant des bananiers « propres » (bananeraie pure) et des bananiers de fermes mélangées,
DeepForest compte-t-il mieux sur Manuel ?

| Rôle | Count banana plants | ReforesTree |
|---|---|---|
| Apprentissage | vols 100_0221, 100_0222 | Flora, Guevara, Arteaga |
| Validation (choix nms / seuil) | vols 100_0223, 100_0224 (pour info) | **Leonor** (c'est lui qui décide) |
| Test | - | **Manuel** |

Nestor (test final) n'est pas utilisé. Le réglage est choisi sur Leonor, une ferme mélangée comme Manuel,
parce qu'un réglage choisi sur la bananeraie pure ne se transfère pas (essai précédent : seuil 0,4, 130 détections sur Manuel).

Avant de lancer : Accelerator **GPU T4**, Internet **On**, Input : `count_banana_v15` et `tuiles_reforestree`.
Résultats : `/kaggle/working/resultats_mixte/`.


In [ ]:
# 1. Installation et repérage des données
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # n'utiliser qu'une des deux cartes graphiques
!pip install -q "deepforest>=2,<3"
from pathlib import Path
import torch
ENTREE = Path('/kaggle/input')
coco = sorted(ENTREE.rglob('_annotations.coco.json'))
tuiles_csv = sorted(ENTREE.rglob('tuiles.csv'))
print(len(coco), 'fichiers COCO trouvés (attendu : 3) | tuiles.csv :', tuiles_csv)
assert len(coco) == 3 and len(tuiles_csv) == 1, "Données manquantes : vérifier les deux datasets en Input"
print('GPU :', torch.cuda.is_available())


In [ ]:
# 2. Préparation : les deux jeux dans un même dossier d'images, un seul fichier d'apprentissage
import json, re, time, shutil
import numpy as np, pandas as pd
from PIL import Image

IMAGES = Path('/kaggle/working/images'); IMAGES.mkdir(parents=True, exist_ok=True)
LABEL = 'Tree'

# 2a. Count banana plants : réduction 3 cm -> 4,5 cm (taille d'origine estimée), boîtes au format DeepForest
PIXEL_ORIGINE, PIXEL_CIBLE = 0.03, 0.045
F = PIXEL_ORIGINE / PIXEL_CIBLE
cb_boites, cb_images = [], []
for fichier in coco:
    j = json.load(open(fichier)); par_image = {}
    for a in j['annotations']:
        par_image.setdefault(a['image_id'], []).append(a['bbox'])
    for im in j['images']:
        nom = im['file_name'].split('_JPG')[0] + '.png'
        img = Image.open(fichier.parent / im['file_name']).convert('RGB')
        img.resize((round(img.width * F), round(img.height * F)), Image.Resampling.LANCZOS).save(IMAGES / nom)
        cb_images.append({'image_path': nom, 'vol': re.match(r'(100_\d+)', nom)[1]})
        for x, y, w, h in par_image.get(im['id'], []):
            if w * F >= 2 and h * F >= 2:
                cb_boites.append({'image_path': nom, 'xmin': round(x * F), 'ymin': round(y * F),
                                  'xmax': round((x + w) * F), 'ymax': round((y + h) * F), 'label': LABEL})
cb_boites, cb_images = pd.DataFrame(cb_boites), pd.DataFrame(cb_images)
cb_app = cb_images[cb_images.vol.isin(['100_0221', '100_0222'])].image_path
cb_val = cb_images[cb_images.vol.isin(['100_0223', '100_0224'])].image_path

# 2b. ReforesTree : tuiles copiées dans le même dossier, bananiers seulement, tuiles sans bananier gardées comme exemples vides
TUILES = tuiles_csv[0].parent
tuiles = pd.read_csv(TUILES / 'tuiles.csv')
ann = pd.read_csv(TUILES / 'annotations.csv')
tuiles['y0'] = tuiles.image_path.str.extract(r'_(\d+)_\d+\.png$')[0].astype(int)
tuiles['x0'] = tuiles.image_path.str.extract(r'_(\d+)\.png$')[0].astype(int)
for n in tuiles.image_path:
    shutil.copy(TUILES / 'images' / n, IMAGES / n)
FERMES_APPRENTISSAGE = ['Flora Pluas RGB', 'Carlos Vera Guevara RGB', 'Carlos Vera Arteaga RGB']
rt_app = tuiles[tuiles.site.isin(FERMES_APPRENTISSAGE)].image_path
rt_bananes = ann[(ann.label == 'banane') & ann.image_path.isin(rt_app)].copy()
rt_bananes = rt_bananes[(rt_bananes.xmax > rt_bananes.xmin) & (rt_bananes.ymax > rt_bananes.ymin)]
rt_bananes['label'] = LABEL
rt_vides = pd.DataFrame({'image_path': sorted(set(rt_app) - set(rt_bananes.image_path)),
                         'xmin': 0, 'ymin': 0, 'xmax': 0, 'ymax': 0, 'label': LABEL})

COLONNES = ['image_path', 'xmin', 'ymin', 'xmax', 'ymax', 'label']
apprentissage = pd.concat([cb_boites[cb_boites.image_path.isin(cb_app)][COLONNES],
                           rt_bananes[COLONNES], rt_vides[COLONNES]])
apprentissage.to_csv('/kaggle/working/apprentissage.csv', index=False)
print(f'Apprentissage : {len(cb_app)} images Count banana + {len(rt_app)} tuiles ReforesTree '
      f'({len(rt_vides)} sans bananier) ; {(apprentissage.xmax > 0).sum()} bananiers au total')

def sans_chevauchement(site):
    t = tuiles[tuiles.site == site]
    return t[(t.y0 % 400 == 0) & (t.x0 % 400 == 0)]
leonor, manuel = sans_chevauchement('Leonor Aspiazu RGB'), sans_chevauchement('Manuel Macias RGB')
print(len(leonor), 'tuiles de validation (Leonor) |', len(manuel), 'tuiles de test (Manuel)')


In [ ]:
# 3. Outils de notation (règle du sujet : centre à moins de 1 m d'un vrai bananier)
from scipy.optimize import linear_sum_assignment

RAYON_M = 1.0
SEUILS_CONFIANCE = np.round(np.arange(0.05, 0.96, 0.05), 2)
SEUILS_NMS = [0.05, 0.2, 0.4]

def centres(df):
    return np.c_[(df.xmin + df.xmax) / 2, (df.ymin + df.ymax) / 2]

def apparier(det, vrai, rayon_px):
    if len(det) == 0 or len(vrai) == 0:
        return 0
    d = np.linalg.norm(det[:, None] - vrai[None], axis=2)
    cout = np.where(d <= rayon_px, d, 1e6)
    l, c = linear_sum_assignment(cout)
    return int((cout[l, c] <= rayon_px).sum())

def grille(modele, noms, verite, taille_pixel_m):
    # noms : images à noter (dans IMAGES) ; verite : boîtes vraies ; taille_pixel_m : un nombre par image
    lignes = []
    for nms in SEUILS_NMS:
        modele.model.nms_thresh = nms
        modele.model.score_thresh = 0.05
        pred = {}
        for nom in noms:
            p = modele.predict_image(path=str(IMAGES / nom))
            pred[nom] = p if p is not None else pd.DataFrame(columns=['xmin', 'ymin', 'xmax', 'ymax', 'score'])
        for s in SEUILS_CONFIANCE:
            vp = nb = 0
            for nom in noms:
                p = pred[nom]; p = p[p.score >= s]
                vp += apparier(centres(p), centres(verite[verite.image_path == nom]), RAYON_M / taille_pixel_m[nom])
                nb += len(p)
            lignes.append({'nms': nms, 'seuil': s, 'detections': nb, 'bananes_vraies': len(verite), 'vp': vp})
    g = pd.DataFrame(lignes)
    g['precision'] = g.vp / g.detections.clip(lower=1)
    g['rappel'] = g.vp / g.bananes_vraies
    g['f1'] = 2 * g.vp / (g.detections + g.bananes_vraies)
    return g


In [ ]:
# 4. Apprentissage (modèle DeepForest d'origine « arbres », 10 époques)
from deepforest import main
try:
    from lightning.pytorch.loggers import CSVLogger
except ImportError:
    from pytorch_lightning.loggers import CSVLogger

EPOQUES = 10
SORTIE = Path('/kaggle/working/resultats_mixte'); SORTIE.mkdir(parents=True, exist_ok=True)

torch.manual_seed(0)
modele = main.deepforest()
modele.load_model('weecology/deepforest-tree')
modele.config.train.csv_file = '/kaggle/working/apprentissage.csv'
modele.config.train.root_dir = str(IMAGES)
modele.config.train.epochs = EPOQUES
modele.config.batch_size = 8
modele.config.workers = 2
journal = CSVLogger('/kaggle/working/journaux', name='mixte')
debut = time.time()
modele.create_trainer(logger=journal)
modele.trainer.fit(modele)
print(f'Apprentissage : {(time.time() - debut) / 60:.1f} min')

# Sécurités : toutes les époques faites ET une loss qui est un vrai nombre à chaque époque
j = pd.read_csv(Path(journal.log_dir) / 'metrics.csv')
pertes = j.dropna(subset=['train_loss_epoch']) if 'train_loss_epoch' in j else j.iloc[0:0]
if j.epoch.max() + 1 < EPOQUES or len(pertes) < EPOQUES or not np.isfinite(pertes.train_loss_epoch).all():
    raise RuntimeError("Apprentissage incomplet ou instable (loss absente ou NaN) : ne pas utiliser ce modèle.")
print(pertes[['epoch', 'train_loss_epoch']].to_string(index=False))
j.to_csv(SORTIE / 'journal.csv', index=False)
torch.save(modele.model.state_dict(), SORTIE / 'poids_mixte.pt')


In [ ]:
# 5. Validation : le réglage (nms, seuil) est choisi sur LEONOR (ferme mélangée), pas sur Manuel
verite_leonor = ann[(ann.label == 'banane') & ann.image_path.isin(leonor.image_path)]
g_leonor = grille(modele, list(leonor.image_path), verite_leonor, dict(zip(leonor.image_path, leonor.taille_pixel_m)))
meilleur = g_leonor.loc[g_leonor.f1.idxmax()]
NMS, SEUIL = meilleur.nms, meilleur.seuil
print(f'Validation Leonor : F1 {meilleur.f1:.3f} (précision {meilleur.precision:.3f}, rappel {meilleur.rappel:.3f}) '
      f'avec nms {NMS}, seuil {SEUIL}')

g_cb = grille(modele, list(cb_val), cb_boites[cb_boites.image_path.isin(cb_val)], {n: PIXEL_CIBLE for n in cb_val})
cb = g_cb[(g_cb.nms == NMS) & (g_cb.seuil == SEUIL)].iloc[0]
print(f'Pour info, vols 0223-0224 de Count banana avec ce réglage : F1 {cb.f1:.3f} '
      f'(meilleur possible {g_cb.f1.max():.3f} ; essai précédent sans ReforesTree : 0.876)')
g_leonor.to_csv(SORTIE / 'grille_leonor.csv', index=False)
g_cb.to_csv(SORTIE / 'grille_count_banana.csv', index=False)


In [ ]:
# 6. Test sur Manuel avec le réglage choisi en 5
verite_manuel = ann[(ann.label == 'banane') & ann.image_path.isin(manuel.image_path)]
g_test = grille(modele, list(manuel.image_path), verite_manuel, dict(zip(manuel.image_path, manuel.taille_pixel_m)))
test = g_test[(g_test.nms == NMS) & (g_test.seuil == SEUIL)].iloc[0]
print(f'MANUEL, réglage choisi sur Leonor : F1 {test.f1:.3f} '
      f'(précision {test.precision:.3f}, rappel {test.rappel:.3f}, {int(test.detections)} détections pour {int(test.bananes_vraies)} bananiers)')
print(f'Pour info, meilleur F1 possible sur Manuel (réglage choisi sur Manuel, donc optimiste) : {g_test.f1.max():.3f}')
print('Comparaison sur Manuel : méthode simple 0.204 | DeepForest ReforesTree seul (essai0) 0.291 | '
      'DeepForest Count banana seul 0.197 (meilleur possible 0.292)')
g_test.to_csv(SORTIE / 'grille_manuel.csv', index=False)
